# 6. 모델 성능 개선과 검증

- 목표: 로지스틱 회귀, 전처리, 스케일링, 규제, 교차 검증을 묶어 성능 개선 흐름을 익힙니다.
- 흐름: 자전거 수요 예측 데이터를 EDA부터 예측 모델링까지 연결하며 전처리, 검증 지표, 교차 검증, 하이퍼파라미터 탐색을 정리합니다.


## 1. 로지스틱 회귀 (Logistic Regression)

### 1.1 개념
- **선형 회귀**는 연속값 예측 → 회귀 문제에 적합  
- **로지스틱 회귀**는 범주형 값 예측 → 분류 문제에 적합  
- 예: 스팸메일(스팸/정상), 시험 결과(합격/불합격)


### 1.2 시그모이드 함수 (Sigmoid Function)
- 로지스틱 회귀의 핵심: 입력값을 **0~1 사이 확률**로 변환  

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

- z = \( w x + b \)  


<img src="image/sigmoid.jpg" alt="sigmoid" width="400">

이미지 출처 : https://en.wikipedia.org/wiki/Sigmoid_function


In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.linspace(-10, 10, 100)  # 일정 간격의 숫자 배열을 만듭니다.
y = sigmoid(x)

plt.plot(x, y)  # 선 그래프를 그립니다.
plt.title("Sigmoid Function")  # 그래프 제목을 설정합니다.
plt.xlabel("z")  # x축 이름을 설정합니다.
plt.ylabel("σ(z)")  # y축 이름을 설정합니다.
plt.grid()  # 그래프 격자를 표시합니다.
plt.show()  # 그래프를 화면에 출력합니다.

### 1.3 결정 경계 (Decision Boundary)
- 시그모이드 함수 결과 ≥ 0.5 → 1 (True)  
- 시그모이드 함수 결과 < 0.5 → 0 (False)  

즉, 확률을 기준으로 분류하는 것.


### 1.4 손실 함수 (Log Loss)

분류 모델은 정답 클래스에 높은 확률을 줄수록 좋은 모델로 봅니다.  
Log Loss는 이 기준을 숫자로 만든 손실 함수입니다.

<details>
<summary>Log Loss 수식 의미 자세히 보기</summary>

- 회귀에서는 MSE 사용  
- 분류에서는 **Log Loss (Cross-Entropy Loss)** 사용  


$L = -\frac{1}{n} \sum_{i=1}^n \Big[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\Big]$


<img src ="image/logx.jpg" width="500">

</details>


### 1.5 Numpy로 구현하기 (선택 심화)

아래 코드는 로지스틱 회귀를 라이브러리 없이 직접 구현한 예시입니다.  
경사하강법으로 `w`, `b`가 어떻게 바뀌는지 확인하고 싶은 경우에만 펼쳐봅니다.

<details>
<summary>로지스틱 회귀 직접 구현 코드 보기</summary>

```python
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

# 데이터: 공부 시간(x) → 합격 여부(y: 0 or 1)
X = np.array([1, 2, 3, 4, 5])  # 넘파이 배열을 만듭니다.
y = np.array([0, 0, 0, 1, 1])  # 넘파이 배열을 만듭니다.

# 파라미터 초기화
w = 0.0
b = 0.0
lr = 0.1
epochs = 1000

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 경사 하강법
for _ in range(epochs):
    z = w*X + b
    y_pred = sigmoid(z)
    
    # 오차
    error = y_pred - y
    
    # 파라미터 업데이트
    dw = np.dot(error, X) / len(X)  # 두 배열의 내적을 계산합니다.
    db = np.sum(error) / len(X)
    
    w -= lr * dw
    b -= lr * db

print("학습된 w:", w)  # 문자열을 정수로 바꿉니다.
print("학습된 b:", b)  # 문자열을 정수로 바꿉니다.

# 예측
test = np.array([2.5, 3.5, 5])  # 넘파이 배열을 만듭니다.
pred = sigmoid(w*test + b)
print("예측 확률:", pred)  # 문자열을 정수로 바꿉니다.
print("분류 결과:", (pred >= 0.5).astype(int))  # 자료형을 변환합니다.
```

</details>


### 1.6 scikit-learn으로 로지스틱 회귀 구현


In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.

X = np.array([1, 2, 3, 4, 5])  # 넘파이 배열을 만듭니다.
y = np.array([0, 0, 0, 1, 1])  # 넘파이 배열을 만듭니다.

X = X.reshape(-1, 1)  # 입력을 2D로 변환
model = LogisticRegression()  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("예측 확률:", model.predict_proba([[3], [4], [5]]))  # 클래스별 예측 확률을 계산합니다.
print("분류 결과:", model.predict([[3], [4], [5]]))  # 학습한 모델로 새 값을 예측합니다.

### 1.7 다중 분류 (Multiclass Classification)
- 로지스틱 회귀는 기본적으로 이진 분류에 사용  
- **OvR(One vs Rest)** 방식으로 다중 분류 확장 가능  
  - 예: 숫자(0~9) 손글씨 분류  



### ✅ 체크포인트
- 로지스틱 회귀는 분류 문제에서 사용되는 지도학습 알고리즘이다.  
- 시그모이드 함수를 사용해 확률(0~1)로 해석할 수 있다.  
- 손실 함수는 Log Loss (교차 엔트로피)를 사용한다.  
- `scikit-learn`으로 손쉽게 이진/다중 분류 문제를 해결할 수 있다.  


## 2. Scikit-learn 모델 사용법 요약

#### 1) 분류(Classification) 예제 — 로지스틱 회귀


In [ ]:
# 분류(Classification) 기본 흐름:
# 1) 더미 데이터 생성 (make_classification)
# 2) 학습/검증 데이터 분리 (train_test_split)
# 3) 모델 생성 및 학습 (LogisticRegression.fit)
# 4) 예측 (predict, predict_proba)
# 5) 성능 평가 (정확도, F1, ROC-AUC, 혼동행렬)``

In [ ]:
from sklearn.datasets import make_classification  # 사이킷런 도구를 불러옵니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.metrics import (  # 사이킷런 도구를 불러옵니다.
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# 1) 더미 데이터 생성
X, y = make_classification(
    n_samples=1000, # - n_samples: 샘플 수
    n_features=10, # - n_features: 특징 수
    n_informative=5, # - n_informative: 실제로 유용한 특징 수
    n_redundant=2, # - n_redundant: 중복 특징 수
    random_state=42 # - random_state: 재현성
)

# 2) 학습/검증 데이터 분리
# - stratify=y: 분류에서는 클래스 비율을 유지하도록 권장
X_train, X_test, y_train, y_test = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

# 3) 모델 생성 및 학습
# - max_iter: 수렴 보장을 위해 여유 있게 설정
# - n_jobs: 가능한 경우 병렬 처리 (LogisticRegression 일부 solver에서만 사용)
clf = LogisticRegression(max_iter=1000)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
clf.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 4) 예측
y_pred = clf.predict(X_test)                    # 라벨 예측(0/1)
y_prob = clf.predict_proba(X_test)[:, 1]        # 양성(1) 클래스 확률

# 5) 성능 평가
acc  = accuracy_score(y_test, y_pred)           # 전체 정확도
prec = precision_score(y_test, y_pred)          # 정밀도(양성 예측의 정확성)
rec  = recall_score(y_test, y_pred)             # 재현율(양성 포착률)
f1   = f1_score(y_test, y_pred)                 # F1(정밀/재현 조화평균)
auc  = roc_auc_score(y_test, y_prob)            # ROC-AUC(임계값 독립 분리도)

cm   = confusion_matrix(y_test, y_pred)         # 혼동행렬
rep  = classification_report(y_test, y_pred)    # 클래스별 정밀/재현/F1 상세

print("[Classification] Logistic Regression")  # 문자열을 정수로 바꿉니다.
print(f"Accuracy  : {acc:.4f}")  # 문자열을 정수로 바꿉니다.
print(f"Precision : {prec:.4f}")  # 문자열을 정수로 바꿉니다.
print(f"Recall    : {rec:.4f}")  # 문자열을 정수로 바꿉니다.
print(f"F1        : {f1:.4f}")  # 문자열을 정수로 바꿉니다.
print(f"ROC-AUC   : {auc:.4f}")  # 문자열을 정수로 바꿉니다.
print("Confusion Matrix:\n", cm)  # 문자열을 정수로 바꿉니다.
print("Classification Report:\n", rep)  # 문자열을 정수로 바꿉니다.


### 기본 지도 학습 알고리즘들 – 실습 문제

### 문제 1. 단순 선형 회귀 (Numpy 구현)
X = [1, 2, 3, 4, 5], y = [2, 4, 6, 8, 10] 데이터를 이용해  
경사 하강법으로 선형 회귀를 학습하고, w와 b를 출력하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

import numpy as np  # 배열 계산을 위한 Numpy입니다.

X = np.array([1, 2, 3, 4, 5])  # 모델에 넣을 입력 데이터입니다.
y = np.array([2, 4, 6, 8, 10])  # 정답 값 또는 두 번째 배열을 준비합니다.

w, b = 0.0, 0.0  # 모델이 학습할 기울기입니다.
lr = 0.01  # 학습률입니다. 한 번에 이동할 크기입니다.
epochs = 1000  # 전체 학습 반복 횟수입니다.

for _ in range(epochs):  # 값을 하나씩 꺼내 반복합니다.
    y_pred = w*X + b  # 모델 예측 결과를 저장합니다.
    error = y_pred - y  # 예측값과 정답의 차이입니다.
    
    dw = (2/len(X)) * np.dot(error, X)  # 기울기 w를 얼마나 바꿀지 계산한 값입니다.
    db = (2/len(X)) * np.sum(error)  # 절편 b를 얼마나 바꿀지 계산한 값입니다.
    
    w -= lr * dw  # 가중치를 손실 감소 방향으로 갱신합니다.
    b -= lr * db  # 절편을 손실 감소 방향으로 갱신합니다.

print("w:", w, "b:", b)  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요

### 문제 2. 선형 회귀 (scikit-learn)
사이킷런의 `LinearRegression`을 사용해, 공부 시간 X=[1,2,3,4,5]과 점수 y=[2,4,6,8,10] 데이터를 학습하고, 6시간 공부했을 때 점수를 예측하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.linear_model import LinearRegression  # 선형 회귀 모델입니다.
import numpy as np  # 배열 계산을 위한 Numpy입니다.

X = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)  # 모델에 넣을 입력 데이터입니다.
y = np.array([2, 4, 6, 8, 10])  # 정답 값 또는 두 번째 배열을 준비합니다.

model = LinearRegression()  # 선형 회귀 모델입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("회귀 계수:", model.coef_)  # 결과를 화면에 출력합니다.
print("절편:", model.intercept_)  # 결과를 화면에 출력합니다.
print("6시간 예측:", model.predict([[6]]))  # 학습한 모델로 새 값을 예측합니다.

```
</details>

In [ ]:
# 여기에 작성하세요
X = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)  # 넘파이 배열을 만듭니다.
y = np.array([2, 4, 6, 8, 10])  # 넘파이 배열을 만듭니다.

### 문제 3. 로지스틱 회귀 (Numpy 구현)
X = [1, 2, 3, 4, 5], y = [0, 0, 0, 1, 1] 데이터에서  
로지스틱 회귀를 경사 하강법으로 학습한 후, X=3, 4, 5에 대한 확률과 분류 결과를 출력하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

import numpy as np  # 배열 계산을 위한 Numpy입니다.

X = np.array([1, 2, 3, 4, 5])  # 모델에 넣을 입력 데이터입니다.
y = np.array([0, 0, 0, 1, 1])  # 정답 값 또는 두 번째 배열을 준비합니다.

w, b = 0.0, 0.0  # 모델이 학습할 기울기입니다.
lr = 0.1  # 학습률입니다. 한 번에 이동할 크기입니다.
epochs = 1000  # 전체 학습 반복 횟수입니다.

def sigmoid(z):  # 함수를 정의합니다.
    return 1 / (1 + np.exp(-z))  # 계산한 값을 함수 밖으로 돌려줍니다.

for _ in range(epochs):  # 값을 하나씩 꺼내 반복합니다.
    z = w*X + b  # 가중합으로 만든 로짓 값입니다.
    y_pred = sigmoid(z)  # 모델 예측 결과를 저장합니다.
    error = y_pred - y  # 예측값과 정답의 차이입니다.
    
    dw = np.dot(error, X) / len(X)  # 기울기 w를 얼마나 바꿀지 계산한 값입니다.
    db = np.sum(error) / len(X)  # 절편 b를 얼마나 바꿀지 계산한 값입니다.
    
    w -= lr * dw  # 가중치를 손실 감소 방향으로 갱신합니다.
    b -= lr * db  # 절편을 손실 감소 방향으로 갱신합니다.

test = np.array([3, 4, 5])  # 넘파이 배열을 만듭니다.
probs = sigmoid(w*test + b)  # 예측 확률을 저장합니다.
preds = (probs >= 0.5).astype(int)  # 자료형을 변환합니다.

print("확률:", probs)  # 결과를 화면에 출력합니다.
print("분류:", preds)  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요

### 문제 4. 로지스틱 회귀 (scikit-learn)
사이킷런의 `LogisticRegression`을 사용해,  
X=[1,2,3,4,5], y=[0,0,0,1,1] 데이터를 학습하고,  
X=3, 4, 5의 예측 확률과 분류 결과를 출력하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 모델입니다.
import numpy as np  # 배열 계산을 위한 Numpy입니다.

X = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)  # 모델에 넣을 입력 데이터입니다.
y = np.array([0, 0, 0, 1, 1])  # 정답 값 또는 두 번째 배열을 준비합니다.

model = LogisticRegression()  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("예측 확률:", model.predict_proba([[3], [4], [5]]))  # 각 클래스에 속할 확률을 예측합니다.
print("분류 결과:", model.predict([[3], [4], [5]]))  # 학습한 모델로 새 값을 예측합니다.

```
</details>



In [ ]:
# 여기에 작성하세요

### ✅ 체크포인트
- 선형 회귀는 연속적인 값을 예측, 로지스틱 회귀는 분류 문제에 사용된다.  
- 경사 하강법은 손실 함수를 줄이면서 파라미터를 학습하는 기본 알고리즘이다.  
- 정규방정식은 소규모 데이터에서 해를 빠르게 구할 수 있다.  
- `scikit-learn`을 사용하면 복잡한 수식을 직접 구현하지 않아도 쉽게 모델을 적용할 수 있다.  


#  머신러닝 더 빠르고 정확하게

머신러닝은 단순히 알고리즘만 아는 것으로 끝나지 않습니다.  
실제 현업에서는 **학습 속도가 느리거나**, **예측 성능이 기대보다 낮은 경우**가 자주 발생합니다.  

👉 이번 토픽은 모델 성능을 최적화하기 위한 핵심 기법들을 다룹니다:  
- 데이터 전처리  
- 규제(Regularization)  
- 모델 평가와 하이퍼파라미터 튜닝  

### 학습 목표

- 다양한 데이터 전처리 기법을 이해하고 적용할 수 있다.  
- 정규화 기법(L1, L2)을 이해하고 활용할 수 있다.  
- 교차 검증과 하이퍼파라미터 튜닝을 통해 모델 성능을 평가하고 개선할 수 있다.

### 목차

#### 1. 들어가기
- 왜 "빠르고 정확하게"가 중요한가?  
- 데이터 전처리 → 정규화 → 모델 평가 & 튜닝으로 이어지는 흐름  


#### 2. 데이터 전처리
- Feature Scaling
  - Normalization (0~1 범위)
  - Standardization (평균=0, 표준편차=1)
  - scikit-learn 실습
- One-hot Encoding
  - 범주형 데이터를 수치형으로 변환
  - pandas `get_dummies()` 실습



#### 3. 규제 (Regularization)
- Bias(편향) vs Variance(분산)  
- Bias-Variance Tradeoff 개념  
- 과적합 방지를 위한 규제 기법
  - L1 규제 (Lasso)
  - L2 규제 (Ridge)
- scikit-learn 실습: Lasso, Ridge 회귀 비교



#### 4. 모델 평가와 하이퍼파라미터 선택
- k겹 교차 검증 (k-Fold Cross Validation)  
  - scikit-learn `cross_val_score` 실습
- 그리드 서치 (Grid Search)  
  - scikit-learn `GridSearchCV` 실습
- 최적의 하이퍼파라미터 찾기  


### ✅ 체크포인트
- 전처리와 정규화는 모델 성능에 직접적으로 영향을 미친다.  
- Regularization은 과적합을 방지하는 핵심 도구이다.  
- 교차 검증과 그리드 서치는 모델 평가와 성능 개선의 표준 절차이다.


## 1. 들어가기

머신러닝 모델을 "더 빠르고 정확하게" 만들기 위해 가장 먼저 해야 할 일은 **데이터 전처리(Data Preprocessing)** 입니다.  
- 현실 데이터는 크기 단위가 제각각 (예: 키=cm, 몸무게=kg, 월수입=만원)  
- 숫자 범위가 다르면, 모델이 특정 특성에 **과도하게 영향을 받음**  
- 따라서 데이터를 적절히 스케일링(Scaling)하고 변환해야 학습이 잘 이루어집니다.


### 예시: 특성 간 범위 차이 문제

데이터에 두 개의 특성이 있다고 합시다:

- $x_1$: **키 비율** → 값의 범위: 0 ~ 1  
- $x_2$: **년 수입** → 값의 범위: 4000 ~ 10000  

### 문제점
- 두 특성을 그대로 사용하면, 모델은 계산 과정에서 **값의 크기가 큰 $x_2$ 1000~2000** 에 더 큰 가중치를 부여하게 됨.  
- 실제로는 $x_1$ (키 비율)도 중요한데, **숫자 스케일 차이 때문에 모델이 무시**할 수 있음.  


### 직관적 비유
- 어떤 학생의 성적을 예로 들어봅시다:
  - **과목 A (출석점수)**: 0~1점  
  - **과목 B (시험점수)**: 1000~2000점  

총점을 단순 합으로 계산하면?  
- 과목 A 점수는 아무리 변해도 1점 차이  
- 과목 B 점수는 최소 1000점 차이  

👉 당연히 **시험점수(과목 B)** 가 모든 결과를 좌우하게 됨 → 출석점수는 사실상 무시됨.  


### 해결 방법
- 데이터를 **스케일링**하여 두 특성이 비슷한 범위를 갖도록 변환해야 함.
- 예:
  - Min-Max Scaling → 모든 값을 0~1 사이로 맞춤  
  - Standardization → 평균=0, 표준편차=1로 변환  

이렇게 하면 모델이 **특성의 실제 중요도**를 제대로 반영할 수 있음.


## 2. 데이터 전처리

### 2.1 Feature Scaling (특성 스케일링)

#### (1) Normalization (정규화)
- 데이터 값을 **0~1 사이로 압축**  
- 공식:  
  $$
  x' = \frac{x - x_{min}}{x_{max} - x_{min}}
  $$


In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
from sklearn.preprocessing import MinMaxScaler  # 0~1 정규화 변환기입니다.

data = np.array([[50], [200], [500]])  # 넘파이 배열을 만듭니다.
scaler = MinMaxScaler()  # 정규화 변환기입니다. 값을 0~1 범위로 맞춥니다.
normalized = scaler.fit_transform(data)  # 기준 학습과 변환을 한 번에 수행합니다.

print("원본 데이터:\n", data)  # 문자열을 정수로 바꿉니다.
print("정규화 데이터:\n", normalized)  # 문자열을 정수로 바꿉니다.

### (2) Standardization (표준화)
- 데이터의 평균=0, 표준편차=1로 맞춤  
- 공식:  
  
  $z = \frac{x - \mu}{\sigma}$

In [ ]:
from sklearn.preprocessing import StandardScaler  # 표준화 변환기입니다.

data = np.array([[50], [200], [500]])  # 넘파이 배열을 만듭니다.
scaler = StandardScaler()  # 표준화 변환기입니다. 평균 0, 표준편차 1로 맞춥니다.
standardized = scaler.fit_transform(data)  # 기준 학습과 변환을 한 번에 수행합니다.

print("표준화 데이터:\n", standardized)  # 문자열을 정수로 바꿉니다.

👉 참고 : 경사 하강법(Gradient Descent) 기반 모델은 **스케일링이 필수적**입니다.  
특성 범위가 다르면, 어떤 방향으로 먼저 학습할지 혼란이 생기기 때문입니다.  

### 2.2 One-hot Encoding (범주형 데이터 변환)

머신러닝 모델은 숫자만 처리할 수 있습니다.  
따라서 범주형 데이터(예: 성별=남/여, 지역=서울/부산/대구)는 **숫자 벡터**로 변환해야 합니다.  

- **문제점**: 단순히 "남=0, 여=1"처럼 하면, **순서/크기 관계가 생겨버림**  
- **해결책**: One-hot Encoding → 각 범주를 별도의 열로 분리, 해당 범주에만 1, 나머지는 0  

### 예시 (Gender 컬럼 변환 전/후)

| Index | Gender |
|-------|--------|
| 0     | Male   |
| 1     | Female |
| 2     | Female |
| 3     | Male   |

👇 One-hot Encoding 적용 후

| Index | Gender_Female | Gender_Male |
|-------|---------------|-------------|
| 0     | 0             | 1           |
| 1     | 1             | 0           |
| 2     | 1             | 0           |
| 3     | 0             | 1           |


In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

df = pd.DataFrame({"Gender": ["Male", "Female", "Female", "Male"]})  # 데이터프레임을 직접 만듭니다.
encoded = pd.get_dummies(df, columns=["Gender"])  # 범주형 값을 원-핫 인코딩합니다.

print(df)  # 문자열을 정수로 바꿉니다.
print(encoded)  # 문자열을 정수로 바꿉니다.

👉 결과:  
- "Male" → [1, 0]  
- "Female" → [0, 1]  

### ✅ 체크포인트
- Normalization: 데이터 범위를 0~1 사이로 맞춤  
- Standardization: 평균=0, 표준편차=1로 맞춤  
- One-hot Encoding: 범주형 데이터를 숫자 벡터로 변환  
- Feature Scaling은 경사 하강법의 효율을 높이고, One-hot Encoding은 범주형 데이터 처리를 가능하게 한다.

## 3. 규제 (Regularization)

### 3.1 왜 규제가 필요한가?
머신러닝 모델은 훈련 데이터에 너무 **과적합(overfitting)** 되거나,  
너무 단순해서 **과소적합(underfitting)** 되는 경우가 많습니다.  

- **과소적합(Underfitting)**: 모델이 단순 → 데이터 패턴을 잘 못 잡음  
- **과적합(Overfitting)**: 모델이 복잡 → 훈련 데이터에는 잘 맞지만 새로운 데이터에서는 성능이 떨어짐  

  <img src="image/overfitting.png" width="500">

이미지 출처 : https://www.geeksforgeeks.org/machine-learning/underfitting-and-overfitting-in-machine-learning/

👉 규제(Regularization)는 **모델이 과적합되는 것을 막고, 일반화 성능을 높이는 방법**입니다.  


### 3.2 규제 개념
규제는 모델이 **불필요하게 큰 가중치**를 가지지 않도록 제약을 주어,  
과적합을 방지하고 일반화 성능을 높이는 방법입니다.


#### L1/L2 규제 (선택 심화)

규제는 모델이 특정 변수에 지나치게 큰 가중치를 주지 않도록 잡아주는 장치입니다.  
L1과 L2의 차이는 "가중치를 어떤 방식으로 줄이느냐"에 있습니다.

<details>
<summary>L1/L2 수식과 예시 보기</summary>

#### **L1 규제 (Lasso Regression)**  
- 가중치의 절댓값 합을 패널티로 부여  
- 일부 가중치를 0으로 만들어 **특성 선택(feature selection)** 효과  

$\text{Loss}_{L1} = \text{MSE} + \lambda \sum_i |w_i|$

예시:

- **규제 전:**  
$ y = 0.2x_1 + 235x_2 + 0.9x_3 $  
- **L1 규제 후:**  
$ y = 0x_1 + 1.3x_2 + 0x_3 $  
→ 일부 가중치가 **완전히 0**이 되어 불필요한 변수가 제거됨

#### **L2 규제 (Ridge Regression)**  
  - 가중치의 제곱합을 패널티로 부여
  - 모든 가중치를 조금씩 줄여 안정적인 모델 생성

  $\text{Loss}_{L2} = \text{MSE} + \lambda \sum_i w_i^2$

예시:

- **규제 전:**  
$y = 0.2x_1 + 235x_2 + 0.9x_3$ 
- **L2 규제 후:**  
$y = 0.1x_1 + 12.3x_2 + 0.5x_3$  
→ 모든 가중치가 **균등하게 작아짐** (0은 되지 않음)

####  **요약**

| 구분 | 패널티 항 | 효과 | 결과 |
|------|------------|--------|--------|
| **L1 규제 (Lasso)** | $\lambda \sum 절대값 w_i $ | 불필요한 변수 제거 | 일부 가중치 0 |
| **L2 규제 (Ridge)** | $\lambda \sum w_i^2$ | 가중치 크기 완화 | 모든 가중치 축소 |

</details>


In [ ]:
## 2.3 scikit-learn으로 과적합 문제 해결

from sklearn.linear_model import LinearRegression, Ridge, Lasso  # 선형 회귀 모델, L2 규제 회귀 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import mean_squared_error  # MSE 회귀 지표 함수입니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

# 샘플 데이터 생성
X = np.random.rand(100, 5) * 10  # 난수를 생성합니다.
y = 3*X[:,0] + 2*X[:,1] - X[:,2] + np.random.randn(100)*2  # 난수를 생성합니다.

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# 선형 회귀
lr = LinearRegression().fit(X_train, y_train)  # 선형 회귀 모델입니다.
print("LinearRegression MSE:", mean_squared_error(y_test, lr.predict(X_test)))  # 학습한 모델로 새 값을 예측합니다.

# Ridge 회귀 (L2)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)  # 릿지 회귀입니다. alpha는 L2 규제 강도입니다.
print("Ridge MSE:", mean_squared_error(y_test, ridge.predict(X_test)))  # 학습한 모델로 새 값을 예측합니다.

# Lasso 회귀 (L1)
lasso = Lasso(alpha=0.1).fit(X_train, y_train)  # 라쏘 회귀입니다. alpha는 L1 규제 강도입니다.
print("Lasso MSE:", mean_squared_error(y_test, lasso.predict(X_test)))  # 학습한 모델로 새 값을 예측합니다.

### 3.4 L1, L2 직접 비교
- **L1 (Lasso)**: 일부 계수=0 → 불필요한 특성 제거 가능  
- **L2 (Ridge)**: 모든 계수를 작게 만들어 안정적인 모델  
- 실제로는 L1+L2 혼합한 **Elastic Net**도 많이 사용  


### ✅ 체크포인트
- 규제는 과적합 방지와 일반화 성능 향상에 핵심적이다.  
- L1(Lasso): 가중치 절댓값 합 → 특성 선택 효과  
- L2(Ridge): 가중치 제곱합 → 안정적인 모델  
- `alpha` 값이 클수록 규제 강도가 세지며, 너무 크면 과소적합 위험이 있다.  


## 4. 모델 평가와 하이퍼파라미터 선택

> 목적: **훈련 데이터에서만 잘 맞는 모델**을 피하고, **새로운 데이터에서도 일관되게 잘 작동(일반화)** 하도록 평가·튜닝한다.


### 4.1 왜 모델 평가가 중요한가?
- **훈련 성능 = 실제 성능 아님**: 훈련 데이터에 맞춘 점수는 낙관적일 수 있음(과적합).
- **일반화 확인**: 보지 못한 데이터(검증/테스트)에서 성능을 확인해야 함.
- **신뢰성**: 평가 절차가 재현 가능해야 하며, 데이터 누수(leakage)를 방지해야 함.


### 4.2 데이터 분할 전략

#### (1) Hold-out 분할(예: **8:1:1 = train:valid:test**)
- **train**: 학습
- **valid**: 하이퍼파라미터 선택/모델 비교
- **test**: 최종 성능 보고(딱 1번만 사용)


In [ ]:
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.

# 데이터 적재
# ------------------------------------------
iris = load_iris()
X, y = iris.data, iris.target

# ------------------------------------------
# 8:1:1 분할 (계층화 분할: 각 클래스 비율 유지)
# ------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("train/valid/test:", X_train.shape, X_valid.shape, X_test.shape)  # 문자열을 정수로 바꿉니다.

#### 변형 예시: **6:2:2**, **7:1.5:1.5** 등. 데이터가 적으면 교차 검증을 권장.

### (2) 시계열(순서형) 데이터
#### 1. 특징
- 데이터는 시간 순서대로 기록됨 (예: 주식 가격, 날씨 데이터, 센서 데이터).
- **순서가 중요**하므로 무작위 섞기(shuffle) 금지.  
- 항상 **과거 → 미래** 순서를 지켜서 학습해야 함.  


#### 2. 올바른 분할 방식 (Sliding Window)
- 일반 데이터 분할(`train_test_split`)은 무작위 추출이 가능하지만, 시계열은 순서를 보존해야 함.  
- **Sliding Window**: 일정 길이의 과거 데이터를 묶어서(train window) 그 직후 미래 구간을 예측(valid window).  
  - 예: "과거 3일 → 다음 1일 예측"  
- 장점: 입력 시퀀스 길이가 일정 → 딥러닝/머신러닝 모델 학습에 바로 활용 가능.

#### 3. 코드 예시: Sliding Window 데이터셋 만들기

In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

# 예제 시계열 데이터 (0 ~ 19)
series = np.arange(20)  # 범위 기반 숫자 배열을 만듭니다.

def make_window_data(series, window=5, horizon=1):
    X, y = [], []
    for i in range(len(series) - window - horizon + 1):
        X.append(series[i:i+window])            # 과거 구간
        y.append(series[i+window:i+window+horizon])  # 예측 구간
    return np.array(X), np.array(y)  # 넘파이 배열을 만듭니다.

# 과거 5일 데이터를 사용해 다음 1일을 예측
X, y = make_window_data(series, window=5, horizon=1)

print("X shape:", X.shape)  # (샘플 수, window 크기)
print("y shape:", y.shape)  # (샘플 수, horizon 크기)
print("첫 번째 샘플 X:", X[0], "-> y:", y[0])  # 문자열을 정수로 바꿉니다.

### 4.3 교차 검증(Cross Validation) 종류
- **KFold**: 무작위로 K분할 → 학습/검증 K회 반복 후 평균.  

- **StratifiedKFold**: 분류에서 **클래스 비율 유지**.
    - **예시**: 암 환자 데이터(환자 10%, 정상인 90%) → 일반 KFold는 어떤 fold엔 환자가 아예 없을 수도 있음.  
- **GroupKFold**: 동일 그룹은 같은 폴드에만 배치.
    - **예시**: 남/녀 를 맞춰야 할때 동일한 사람의 사진이 2장이상 존재할때 같은 그룹(train/val/test)에 배치
- **TimeSeriesSplit**: 시계열 전용(시간 순서 유지).


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold  # 교차검증 점수 함수입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

iris = load_iris()

# 입력 특성 행렬(X)과 정답 벡터(y) 분리
# X : 꽃받침 길이, 꽃받침 폭, 꽃잎 길이, 꽃잎 폭 (4개의 특성)
# y : 붓꽃 품종 (0=setosa, 1=versicolor, 2=virginica)
X, y = iris.data, iris.target

# 로지스틱 회귀 모델 생성
# 분류(classification) 문제에 사용되는 선형 모델
# max_iter=200 : 학습 반복 횟수 제한 (기본값은 100, 수렴 문제 방지를 위해 늘림)
model = LogisticRegression(max_iter=200)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.

# StratifiedKFold : 각 클래스 비율(레이블 분포)을 유지하면서 데이터셋을 여러 조각으로 나누는 K-겹 교차검증 방법
# n_splits=5 → 데이터를 5개의 폴드(fold)로 나눔 (즉, 5번의 학습/평가 수행)
# shuffle=True → 데이터를 무작위로 섞은 후 분할 (데이터 순서에 의한 편향 방지)
# random_state=42 → 난수 시드를 고정하여 재현성 확보
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")  # 교차검증 점수를 계산합니다.

# cross_val_score() : 교차 검증을 자동으로 수행해주는 함수
# 인자 설명:
#   model   : 사용할 학습 모델 (여기서는 로지스틱 회귀)
#   X, y    : 입력 데이터와 정답 레이블
#   cv      : 교차검증 분할 전략 (StratifiedKFold 객체)
#   scoring : 성능 평가 지표. 여기서는 'accuracy' (정확도)
#
# 실행 과정:
#   1. 데이터를 StratifiedKFold에 따라 5개의 폴드로 나눔
#   2. 각 폴드에 대해 1개는 테스트, 나머지 4개는 학습용으로 사용
#   3. 5번 반복하여 모델의 정확도를 계산
#   4. 각 반복에서의 정확도 점수를 배열 형태로 반환
print("교차 검증 점수:", scores)  # 문자열을 정수로 바꿉니다.
print("평균 정확도:", np.mean(scores))  # 평균을 계산합니다.

In [ ]:
# 각 폴드의 인덱스 확인
for fold_idx, (train_index, test_index) in enumerate(cv.split(X, y), start=1):
    print(f"\n📂 Fold {fold_idx}")  # 문자열을 정수로 바꿉니다.
    print(f"학습 데이터 인덱스 ({len(train_index)}개): {train_index[:10]} ...")  # 앞부분만 표시
    print(f"테스트 데이터 인덱스 ({len(test_index)}개): {test_index[:10]} ...")  # 문자열을 정수로 바꿉니다.
    print(f"→ 테스트 데이터의 클래스 분포: {np.bincount(y[test_index])}")  # 문자열을 정수로 바꿉니다.

#### **TIP**: 스케일링 기준은 훈련 데이터에서만 학습하고, 검증/테스트 데이터에는 변환만 적용합니다.


### 4.4 평가 지표 선택 가이드


#### 1) 분류(Classification)

#### 혼동행렬(Confusion Matrix) 표

| 실제 \ 예측 | 0 (Negative)         | 1 (Positive)         |
|-------------|-----------------------|-----------------------|
| **0 (Negative)** | **TN**: 진짜 음성 (정상 정답) | **FP**: 거짓 양성 (거짓 경보) |
| **1 (Positive)** | **FN**: 거짓 음성 (놓침)     | **TP**: 진짜 양성 (정상 검출) |

#### 혼동행렬(Confusion Matrix) 용어 정리
- **TP (True Positive)**: 실제 1이고, 예측도 1  
- **FP (False Positive)**: 실제 0인데, 예측이 1 (거짓 경보)  
- **TN (True Negative)**: 실제 0이고, 예측도 0  
- **FN (False Negative)**: 실제 1인데, 예측이 0 (놓침)

#### **지표 공식**  
 - Accuracy = (TP + TN) / (TP + FP + TN + FN)  
 - Precision = TP / (TP + FP)  
 - Recall = TP / (TP + FN)  
 - F1 = 2 · (Precision · Recall) / (Precision + Recall)

| 지표 | 정의/설명 | 장점 | 주의할 점 / 활용 상황 |
|------|-----------|------|------------------------|
| **Accuracy** | 전체 샘플 중 정답 비율 | 직관적, 해석 쉬움 | 클래스 불균형(예: 정상 99%, 이상 1%)에 매우 취약 |
| **Precision (정밀도)** | `예측=양성` 중 실제 양성 비율 | 잘못된 경보(오탐) 줄이는 데 중요 | 양성 놓침(미탐)에는 둔감 |
| **Recall (재현율)** | 실제 양성 중 예측=양성 비율 | 양성 놓치지 않는 게 중요할 때 유리 | 오탐(거짓 양성)이 많아질 수 있음 |
| **F1 Score** | Precision·Recall의 조화 평균 | Precision·Recall 균형 평가 | 해석은 다소 어렵지만 불균형 데이터에 자주 사용 |
| **ROC-AUC** | 모든 임계값에서 TPR vs FPR 곡선 아래 면적 | 임계값에 독립적, 전반적 성능 평가 | 클래스 극심 불균형일 때 과대평가될 수 있음 |
| **PR-AUC** | Precision-Recall 곡선 아래 면적 | 양성 클래스 희소할 때 유리 | ROC-AUC보다 해석 어렵지만 불균형 심할 때 필수 |
| **Top-k Accuracy** | 다중 클래스에서 상위 k개 예측 안에 정답 있는지 | 이미지 분류(예: Top-5 Accuracy)에서 유리 | k 설정 필요 |

### 2) 회귀(Regression)

| 지표 | 정의/설명 | 장점 | 주의할 점 / 활용 상황 |
|------|-----------|------|------------------------|
| **MAE (Mean Absolute Error)** | 절댓값 오차 평균 | 해석 직관적, 이상치 영향 적음 | 큰 오차에 둔감 |
| **MSE (Mean Squared Error)** | 제곱 오차 평균 | 미분/최적화에 유리 | 큰 오차에 과도한 패널티 |
| **RMSE (Root Mean Squared Error)** | 제곱 오차의 제곱근 | 원 단위 복원, 큰 오차 강조 | MAE보다 이상치 민감 |
| **R² (결정계수)** | 모델이 데이터 분산을 얼마나 설명하는지 (1=완벽) | 상대적 성능 비교에 유용 | 데이터 분포에 따라 음수가 될 수 있음 |

In [ ]:
# scoring 예시: "accuracy", "f1_macro", "roc_auc_ovr", "neg_mean_absolute_error" 등

### 4.5 하이퍼파라미터란?
- **파라미터**: 모델이 **데이터로부터 학습**하는 값 (예: 모델의 가중치)
- **하이퍼파라미터**: 사람이 **사전에 설정**하는 값 (예: 정규화 강도 C/alpha, 트리 깊이, 학습률)

| 알고리즘 | 대표 하이퍼파라미터 | 의미/영향 |
|---|---|---|
| LogisticRegression | C, penalty, solver | 정규화 강도, 규제 형태 |
| SVM | C, kernel, gamma | 마진/복잡도 제어, 커널 폭 |
| RandomForest | n_estimators, max_depth, min_samples_split | 앙상블 크기/복잡도 |
| Ridge/Lasso | alpha | L2/L1 정규화 강도 |
| XGBoost/LightGBM | n_estimators, learning_rate, max_depth, subsample | 부스팅 단계/학습률/복잡도 |


### 4.6 그리드 서치(Grid Search)

Grid Search는 모델의 하이퍼파라미터 후보를 미리 정해두고, 교차 검증으로 모든 조합을 비교해 가장 좋은 조합을 찾는 방법입니다.

예를 들어 로지스틱 회귀에서는 `C` 값을 바꾸면 규제 강도가 달라집니다. 어떤 값이 좋은지는 데이터마다 다르기 때문에 여러 후보를 비교합니다.

```python
from sklearn.model_selection import GridSearchCV
```

기본 흐름은 다음과 같습니다.

1. 사용할 모델을 정합니다.
2. 바꿔볼 하이퍼파라미터 후보를 딕셔너리로 만듭니다.
3. `GridSearchCV`로 교차 검증을 수행합니다.
4. `best_params_`, `best_score_`로 결과를 확인합니다.

데이터 누수를 피하려면 전처리 기준을 훈련 데이터에서만 학습해야 합니다. 검증이나 테스트 데이터는 모델을 평가할 때만 사용합니다.


In [ ]:
from sklearn.datasets import load_iris  # 예제 데이터셋입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.model_selection import GridSearchCV, StratifiedKFold  # 후보 조합 교차검증 도구입니다.

X, y = load_iris(return_X_y=True)  # 붓꽃 데이터를 입력과 정답으로 나눕니다.

model = LogisticRegression(max_iter=200)  # 비교할 기본 모델입니다.

param_grid = {  # 비교할 하이퍼파라미터 후보입니다.
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],
    "solver": ["liblinear", "lbfgs"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 클래스 비율을 유지하며 5겹으로 나눕니다.

grid = GridSearchCV(  # 모든 후보 조합을 교차 검증으로 비교합니다.
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

grid.fit(X, y)  # 후보 조합을 비교하며 모델을 학습합니다.

print("최적 하이퍼파라미터:", grid.best_params_)
print("교차 검증 평균 점수:", grid.best_score_)


### 핵심 정리

- **GridSearchCV**: 하이퍼파라미터 후보를 모두 비교해 좋은 조합을 찾습니다.
- **cv**: 데이터를 몇 겹으로 나누어 검증할지 정합니다.
- **scoring**: 어떤 지표로 좋은 모델을 판단할지 정합니다.
- 후보가 너무 많으면 시간이 오래 걸리므로, 처음에는 적은 후보로 시작합니다.


### 연습문제. GridSearchCV의 학습 횟수 계산  

위 코드는 로지스틱 회귀(Logistic Regression)를 교차검증과 함께 수행하는 코드입니다.  
이때 모델 학습이 총 몇 번 일어나는지 계산하세요.  

 참고:  
- 교차검증: StratifiedKFold(n_splits=5)  
- 파라미터 조합: C(5개) × penalty(1개) × solver(2개)

총 학습 횟수를 구하세요.  


<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

# 파라미터 조합 수 = 5 × 1 × 2 = 10
# 교차검증 fold 수 = 5
# 교차검증 중 학습 횟수 = 10 × 5 = 50
# 최적 조합으로 전체 훈련 세트 재학습 = 1회 추가

total_fits = 50 + 1  # 교차검증 50회와 최종 재학습 1회를 더합니다.
print("총 학습 횟수:", total_fits, "회")  # 계산한 총 학습 횟수를 출력합니다.

```
</details>


## 4.7 추가 팁 & 흔한 함정

- **Seed 고정**:  
  매번 실행할 때마다 결과가 달라지지 않게,  `random_state` 값을 고정하세요.  
  → 실험을 **다시 해도 같은 결과(재현성)** 가 나옵니다.

- **클래스 불균형**:  
  데이터에서 한 클래스(예: 0, 1)가 너무 많거나 적을 때는 `stratify` 옵션을 사용해 비율을 맞추고,  
  모델 학습 시 `class_weight='balanced'`를 주면 좋습니다. `-> 다수 클래스 예측시 패널티 부여`  
  → 평가 지표도 정확도(accuracy) 대신 **F1-score**나 **PR-AUC**가 더 적절합니다.

- **베이스라인**:  
  처음부터 복잡한 모델을 쓰기보다,  
  간단한 모델(예: 로지스틱 회귀, 선형 회귀)로 먼저 기준 점수를 만들어보세요.  
  → 이후 복잡한 모델(XGBoost, 딥러닝 등)이 정말 도움이 되는지 확인할 수 있습니다.

- **RandomizedSearchCV**:  
  하이퍼파라미터 후보가 너무 많을 때, 전부 다 시도(GridSearch)하는 대신 **일부만 랜덤하게 탐색**하는 게 효율적입니다.  
  → 빠르지만 충분히 좋은 조합을 찾을 수 있습니다.

- **특성 스케일링**:  
  SVM, KNN, 로지스틱 회귀처럼 **거리나 크기를 계산하는 모델**에서는 각 특성(컬럼)의 단위를 맞춰주는 게 중요합니다.  
  → `StandardScaler`(평균 0, 표준편차 1)나 `MinMaxScaler`(0~1 정규화)를 꼭 사용하세요.

- **데이터 누수(Data Leakage)**:  
  스케일링, 인코딩, 특성 선택 등은 **훈련 데이터에만** 맞춰서 해야 합니다.  
  검증/테스트 데이터까지 포함해 전처리 기준을 학습하면 평가 점수가 실제보다 좋게 보일 수 있습니다.


### ✅ 체크포인트
- [ ] 8:1:1 분할 또는 교차 검증으로 **일반화 성능**을 확인했는가?  
- [ ] 전처리 기준을 **훈련 데이터에만** 맞춰 데이터 누수를 방지했는가?  
- [ ] 문제 특성에 맞는 **평가지표**를 선택했는가?  
- [ ] 합리적인 **하이퍼파라미터 공간**을 정의했는가? 


## 자전거 수요 예측: EDA부터 모델링까지

자전거 대여량(`count`)을 예측하는 회귀 문제입니다. 데이터를 바로 모델에 넣기보다, 먼저 데이터 구조와 패턴을 확인하고 그 결과를 특성 설계와 검증 방식에 연결합니다.

전체 흐름은 다음과 같습니다.
- 데이터 구조 확인
- EDA로 수요 패턴 파악
- 예측에 사용할 특성 만들기
- 학습/검증 데이터 분리
- baseline 모델 평가
- 모델 개선과 하이퍼파라미터 탐색
- 예측 결과 점검


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

train_raw = pd.read_csv("자전거수요예측/train.csv")
test_raw = pd.read_csv("자전거수요예측/test.csv")

train_raw["datetime"] = pd.to_datetime(train_raw["datetime"])
test_raw["datetime"] = pd.to_datetime(test_raw["datetime"])

print(train_raw.shape)
train_raw.head()


### 1. 데이터 구조 확인

먼저 행과 열, 결측치, 데이터 타입을 확인합니다. 이 단계에서 타깃 컬럼과 사용하면 안 되는 컬럼도 함께 구분합니다.

- `count`: 예측해야 하는 전체 대여량
- `casual`, `registered`: `count`를 구성하는 값이므로 예측 입력으로 쓰면 안 됩니다.
- `datetime`: 그대로 쓰기보다 시간, 요일, 월 같은 파생 변수로 바꿔 사용합니다.


In [ ]:
print("데이터 크기")
print("train:", train_raw.shape)
print("test :", test_raw.shape)

print()
print("컬럼")
print(train_raw.columns.tolist())

print()
print("결측치")
print(train_raw.isna().sum())

print()
print("타깃 기초 통계")
print(train_raw["count"].describe())


### 2. 타깃 분포 확인

대여량은 0보다 작을 수 없는 값이고, 특정 시간대에 큰 값이 몰릴 수 있습니다. 분포가 한쪽으로 치우쳐 있으면 큰 값의 오차가 모델 학습에 크게 작용합니다.

`log1p(count)`는 `count`에 1을 더한 뒤 로그를 취한 값입니다. 값의 치우침을 완화해 회귀 모델이 더 안정적으로 학습되는 경우가 많습니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_raw["count"], bins=40, color="#4C78A8")
axes[0].set_title("Count distribution")
axes[0].set_xlabel("count")
axes[0].set_ylabel("Frequency")

axes[1].hist(np.log1p(train_raw["count"]), bins=40, color="#F58518")
axes[1].set_title("Log1p count distribution")
axes[1].set_xlabel("log1p(count)")

plt.tight_layout()
plt.show()


### 3. 시간 특성 만들기

자전거 대여량은 시간의 영향을 많이 받습니다. 출퇴근 시간, 주말 여부, 계절에 따라 수요가 달라질 수 있으므로 `datetime`에서 모델이 이해할 수 있는 숫자형 특성을 만듭니다.


In [ ]:
def add_datetime_features(df):
    df = df.copy()
    dt = pd.to_datetime(df["datetime"])
    df["year"] = dt.dt.year
    df["month"] = dt.dt.month
    df["day"] = dt.dt.day
    df["hour"] = dt.dt.hour
    df["dayofweek"] = dt.dt.dayofweek
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    return df

train_eda = add_datetime_features(train_raw)
test_eda = add_datetime_features(test_raw)

train_eda[["datetime", "year", "month", "day", "hour", "dayofweek", "is_weekend"]].head()


### 4. 시간별 수요 패턴

시간대, 요일, Average rentals by month을 보면 어떤 특성이 예측에 필요할지 판단할 수 있습니다. 그래프는 모델링 전에 가설을 세우는 도구입니다.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train_eda.groupby("hour")["count"].mean().plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Average rentals by hour")
axes[0].set_xlabel("hour")
axes[0].set_ylabel("mean count")

train_eda.groupby("dayofweek")["count"].mean().plot(kind="bar", ax=axes[1], color="#54A24B")
axes[1].set_title("Average rentals by day of week")
axes[1].set_xlabel("dayofweek")

train_eda.groupby("month")["count"].mean().plot(kind="bar", ax=axes[2], color="#E45756")
axes[2].set_title("Average rentals by month")
axes[2].set_xlabel("month")

plt.tight_layout()
plt.show()


### 5. 날씨와 수요 패턴

자전거 수요는 날씨와 기온에도 영향을 받습니다. 평균 비교와 산점도를 함께 보면 범주형 변수와 연속형 변수를 각각 어떻게 다룰지 감을 잡을 수 있습니다.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train_eda.groupby("season")["count"].mean().plot(kind="bar", ax=axes[0], color="#72B7B2")
axes[0].set_title("Average rentals by season")
axes[0].set_xlabel("season")
axes[0].set_ylabel("mean count")

train_eda.groupby("weather")["count"].mean().plot(kind="bar", ax=axes[1], color="#B279A2")
axes[1].set_title("Average rentals by weather")
axes[1].set_xlabel("weather")

axes[2].scatter(train_eda["temp"], train_eda["count"], alpha=0.25, s=10, color="#F58518")
axes[2].set_title("Temperature vs count")
axes[2].set_xlabel("temp")
axes[2].set_ylabel("count")

plt.tight_layout()
plt.show()


### 6. 모델링 방향 정리

EDA에서 확인한 내용을 모델 입력으로 바꿉니다.

- 시간 패턴이 강하므로 `hour`, `dayofweek`, `month`, `is_weekend`를 사용합니다.
- `season`, `weather`, `hour`, `dayofweek`, `month`는 범주처럼 해석할 수 있어 원-핫 인코딩합니다.
- `casual`, `registered`는 정답을 쪼갠 값이라 입력 변수에서 제외합니다.
- 타깃은 `log1p(count)`로 학습하고, 예측 후 `expm1`로 원래 단위로 되돌립니다.


In [ ]:
base_features = [
    "holiday", "workingday", "temp", "atemp", "humidity", "windspeed",
    "year", "day", "is_weekend"
]
categorical_features = ["season", "weather", "month", "hour", "dayofweek"]

X_all = pd.get_dummies(
    train_eda[base_features + categorical_features],
    columns=categorical_features,
    dtype=int
)
X_test = pd.get_dummies(
    test_eda[base_features + categorical_features],
    columns=categorical_features,
    dtype=int
)
X_test = X_test.reindex(columns=X_all.columns, fill_value=0)

y_all = train_eda["count"]
y_log = np.log1p(y_all)

print("학습 입력:", X_all.shape)
print("테스트 입력:", X_test.shape)
X_all.head()


### 7. 학습/검증 데이터 분리

시간이 있는 데이터는 무작위로 섞기보다 앞부분으로 학습하고 뒤쪽 데이터로 검증하면 실제 예측 상황에 더 가깝습니다. 여기서는 날짜 순서 기준으로 앞 80%를 학습, 뒤 20%를 검증으로 사용합니다.


In [ ]:
ordered_index = train_eda.sort_values("datetime").index
split_point = int(len(ordered_index) * 0.8)

train_index = ordered_index[:split_point]
valid_index = ordered_index[split_point:]

X_train = X_all.loc[train_index]
X_valid = X_all.loc[valid_index]
y_train = y_all.loc[train_index]
y_valid = y_all.loc[valid_index]
y_train_log = np.log1p(y_train)

print("train:", X_train.shape)
print("valid:", X_valid.shape)
print(train_eda.loc[train_index, "datetime"].min(), "~", train_eda.loc[train_index, "datetime"].max())
print(train_eda.loc[valid_index, "datetime"].min(), "~", train_eda.loc[valid_index, "datetime"].max())


### 8. 평가 지표

회귀 문제에서는 예측값과 실제값의 차이를 봅니다.

- MAE: 평균적으로 얼마나 틀렸는지 보기 쉽습니다.
- RMSE: 큰 오차에 더 민감합니다.
- RMSLE: Actual vs predicted의 비율 차이를 보는 데 유용합니다.
- R²: 평균값으로 예측하는 것보다 얼마나 잘 설명하는지 확인합니다.


In [ ]:
def regression_report(y_true, pred, name):
    pred = np.clip(pred, 0, None)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "RMSLE": np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(pred))),
        "R2": r2_score(y_true, pred),
    }

results = []


### 9. Baseline 모델

먼저 아주 단순한 기준선을 만듭니다. 평균값만 예측하는 모델보다 좋아야 실제 모델링의 의미가 있습니다.


In [ ]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_valid)

results.append(regression_report(y_valid, dummy_pred, "Mean baseline"))
pd.DataFrame(results)


### 10. Ridge 회귀

선형 모델은 빠르고 해석이 쉽습니다. 스케일 차이가 큰 특성이 섞여 있으므로 표준화한 뒤 학습합니다.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

ridge = Ridge(alpha=10.0)
ridge.fit(X_train_scaled, y_train_log)
ridge_pred = np.expm1(ridge.predict(X_valid_scaled))

results.append(regression_report(y_valid, ridge_pred, "Ridge"))
pd.DataFrame(results)


### 11. 랜덤포레스트 회귀

랜덤포레스트는 비선형 패턴을 잘 잡습니다. 시간대와 날씨처럼 서로 결합되어 영향을 주는 특성이 있을 때 선형 모델보다 좋은 성능을 내는 경우가 많습니다.


In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train_log)
rf_pred = np.expm1(rf.predict(X_valid))

results.append(regression_report(y_valid, rf_pred, "RandomForest"))
pd.DataFrame(results).sort_values("RMSE")


### 12. 예측 결과 점검

평가 지표만 보면 어떤 구간에서 틀리는지 알기 어렵습니다. Actual vs predicted의 관계, 시간대별 평균 예측을 함께 확인합니다.


In [ ]:
valid_check = train_eda.loc[valid_index, ["datetime", "hour", "count"]].copy()
valid_check["pred"] = np.clip(rf_pred, 0, None)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(valid_check["count"], valid_check["pred"], alpha=0.25, s=10, color="#4C78A8")
axes[0].plot([0, valid_check["count"].max()], [0, valid_check["count"].max()], color="black", linewidth=1)
axes[0].set_title("Actual vs predicted")
axes[0].set_xlabel("actual")
axes[0].set_ylabel("predicted")

valid_check.groupby("hour")[["count", "pred"]].mean().plot(ax=axes[1])
axes[1].set_title("Hourly actual vs predicted mean")
axes[1].set_xlabel("hour")
axes[1].set_ylabel("mean count")

plt.tight_layout()
plt.show()


### 13. 하이퍼파라미터 탐색

랜덤포레스트의 성능은 트리 수, 깊이, 리프 노드 조건에 영향을 받습니다. 전체 후보를 모두 시도하기보다 작은 후보 범위에서 무작위 탐색으로 시작합니다.

시간 순서를 가진 데이터이므로 교차 검증도 앞 데이터로 학습하고 뒤 데이터로 검증하는 `TimeSeriesSplit`을 사용합니다.


In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [8, 12, 16, None],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.7, 1.0],
}

tscv = TimeSeriesSplit(n_splits=3)

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=6,
    cv=tscv,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)
search.fit(X_train, y_train_log)

print("최적 파라미터:", search.best_params_)
print("교차 검증 RMSLE:", np.sqrt(-search.best_score_))


In [ ]:
tuned_rf = search.best_estimator_
tuned_pred = np.expm1(tuned_rf.predict(X_valid))

results.append(regression_report(y_valid, tuned_pred, "Tuned RandomForest"))
pd.DataFrame(results).sort_values("RMSE")


### 14. 특성 중요도

트리 기반 모델은 어떤 특성이 예측에 많이 쓰였는지 확인할 수 있습니다. 중요도가 높다고 원인이라는 뜻은 아니지만, 모델이 어느 정보를 주로 참고했는지 보는 데 도움이 됩니다.


In [ ]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": tuned_rf.feature_importances_
}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(8, 5))
plt.barh(importance["feature"][::-1], importance["importance"][::-1], color="#4C78A8")
plt.title("Top feature importance")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

importance


### 15. 제출 파일 만들기와 결과 확인

검증 데이터에서 모델 성능을 확인한 뒤, 검증 결과가 가장 안정적인 설정으로 전체 학습 데이터를 다시 학습합니다. Kaggle 제출 파일은 `datetime`, `count` 두 컬럼으로 만듭니다.

제출 전에는 검증 결과표를 다시 확인합니다. 자전거 수요 예측 대회는 RMSLE가 낮을수록 좋은 점수입니다. 하이퍼파라미터 탐색 결과가 항상 더 좋은 것은 아니므로, 검증표를 보고 제출 모델을 선택합니다.


In [ ]:
result_table = pd.DataFrame(results).drop_duplicates("model", keep="last").sort_values("RMSE")
print("검증 결과")
print(result_table)

final_model = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X_all, y_log)

test_pred = np.expm1(final_model.predict(X_test))
test_pred = np.clip(test_pred, 0, None)

bike_submission = pd.DataFrame({
    "datetime": test_eda["datetime"],
    "count": test_pred,
})

bike_submission.to_csv("bike_submission.csv", index=False)

print()
print("제출 모델: RandomForest")
print("제출 파일: bike_submission.csv")
print("제출 행 수:", len(bike_submission))
bike_submission.head()


### 체크포인트

- EDA는 그래프를 많이 그리는 것이 목적이 아니라, 모델에 넣을 특성과 검증 방식을 정하는 과정입니다.
- 시간형 데이터는 무작위 분할보다 시간 순서를 고려한 검증이 더 자연스러운 경우가 많습니다.
- `casual`, `registered`처럼 정답을 직접 구성하는 컬럼은 입력 변수에서 제외합니다.
- 회귀 평가는 MAE, RMSE, RMSLE처럼 오차를 여러 관점에서 함께 봅니다.
- 하이퍼파라미터 탐색은 검증 전략이 먼저 정해진 뒤에 진행해야 합니다.
